<a href="https://colab.research.google.com/github/motunrayoade333/Motunrayo-s-portfolio/blob/main/Week_4_File_Handling_%26_Data_Quality_Assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
from datetime import datetime, date

with open("supply_chain_dataset.csv", newline="") as file:
    reader = csv.DictReader(file)
    columns = reader.fieldnames
    rows = list(reader)

def parse_date(text):
    if "/" in text:                      # 11/28/2022
        return datetime.strptime(text, "%m/%d/%Y").date()
    return date.fromisoformat(text)      # 2022-02-20

print("Rows:", len(rows))

Rows: 3000


In [2]:
# 1. Missing values: empty cells per column
print("MISSING VALUES")
for col in columns:
    empty = sum(1 for row in rows if row[col].strip() == "")
    print(f"  {col}: {empty}")

# 2. Exact duplicate rows (a set remembers what we've already seen)
seen = set()
duplicates = 0
for row in rows:
    key = tuple(row.values())
    if key in seen:
        duplicates += 1
    seen.add(key)
print("\nExact duplicate rows:", duplicates)

# 3. Text inconsistencies: distinct values
print("\nDISTINCT VALUES")
for col in ["supplier_name", "warehouse_destination", "delivery_status", "product_category"]:
    print(f"  {col} ({len({row[col] for row in rows})}):", sorted({row[col] for row in rows}))

extra_spaces = sum(1 for row in rows if row["product_category"] != row["product_category"].strip())
print("\nCategory values with extra spaces:", extra_spaces)

# 4. Date formats
print("\nDATE FORMATS")
for col in ["order_date", "delivery_date"]:
    slash = sum(1 for row in rows if "/" in row[col])
    print(f"  {col}: MM/DD/YYYY = {slash}, YYYY-MM-DD = {len(rows) - slash}")

MISSING VALUES
  order_id: 0
  order_date: 0
  product_category: 0
  quantity: 0
  unit_price: 0
  total_cost: 0
  supplier_name: 0
  supplier_location: 0
  lead_time_days: 0
  warehouse_destination: 0
  delivery_date: 0
  actual_delivery_days: 0
  delivery_status: 0
  on_time: 0
  product_defects: 154
  return_status: 146

Exact duplicate rows: 0

DISTINCT VALUES
  supplier_name (15): ['Supplier A', 'Supplier B', 'Supplier C', 'Supplier D', 'Supplier E', 'SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE', 'suppliera', 'supplierb', 'supplierc', 'supplierd', 'suppliere']
  warehouse_destination (12): ['WH_1', 'WH_2', 'WH_3', 'WH_4', 'WH_5', 'WH_6', 'Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
  delivery_status (6): ['Cancelled', 'Delayed', 'Delievered', 'Delivered', 'In-Transit', 'Pendng']
  product_category (8): ['Connectors', 'Control Systems', 'Fasteners', 'Hydraulics', 'Motors', 'Pumps', 'Sensors', 'Valves']

Category values 

In [3]:
# Task 2: Logical checks

skip_statuses = ["Cancelled", "In-Transit", "Pending", "Pendng"]   # 'Pendng' is the typo in the raw file

failures = {
    "cost_zero_or_negative": [],
    "cost_not_qty_x_price": [],
    "delivered_before_ordered": [],
    "actual_days_mismatch": [],
    "on_time_disagrees": [],
}
skipped = 0

for row in rows:
    order_id = row["order_id"]
    cost = float(row["total_cost"])
    qty = float(row["quantity"])
    price = float(row["unit_price"])

    # Checks that apply to every row
    if cost <= 0:
        failures["cost_zero_or_negative"].append(order_id)
    if abs(qty * price - cost) > 1:
        failures["cost_not_qty_x_price"].append(order_id)

    # Delivery checks: skip orders that have no delivery yet
    if row["delivery_status"].strip() in skip_statuses:
        skipped += 1
        continue

    ordered = parse_date(row["order_date"])
    delivered = parse_date(row["delivery_date"])
    days_between = (delivered - ordered).days
    actual = int(float(row["actual_delivery_days"]))
    lead = float(row["lead_time_days"])

    if delivered < ordered:
        failures["delivered_before_ordered"].append(order_id)
    if days_between != actual:
        failures["actual_days_mismatch"].append(order_id)

    late = actual > lead
    if (row["on_time"] == "Yes" and late) or (row["on_time"] == "No" and not late):
        failures["on_time_disagrees"].append(order_id)

print("LOGICAL CHECKS")
for check, ids in failures.items():
    print(f"  {check}: {len(ids)} rows, e.g. {ids[:3]}")
print("\nOrders skipped for delivery checks:", skipped)

LOGICAL CHECKS
  cost_zero_or_negative: 1 rows, e.g. ['ORD-1521']
  cost_not_qty_x_price: 154 rows, e.g. ['ORD-0029', 'ORD-0034', 'ORD-0057']
  delivered_before_ordered: 23 rows, e.g. ['ORD-0007', 'ORD-0060', 'ORD-0122']
  actual_days_mismatch: 23 rows, e.g. ['ORD-0007', 'ORD-0060', 'ORD-0122']
  on_time_disagrees: 54 rows, e.g. ['ORD-0033', 'ORD-0039', 'ORD-0081']

Orders skipped for delivery checks: 107


In [5]:
# Unique problem rows (one row can fail several checks)
logic_ids = set()
for ids in failures.values():
    logic_ids |= set(ids)

blank_ids = {row["order_id"] for row in rows
             if row["product_defects"] == "" or row["return_status"] == ""}

print("Rows with a logical problem:", len(logic_ids))
print("Rows with a blank:", len(blank_ids))
print("Rows with any problem:", len(logic_ids | blank_ids))

Rows with a logical problem: 228
Rows with a blank: 293
Rows with any problem: 498


In [4]:
# Task 3: Standardise and save

def standardise_supplier(name):
    """'Supplier A', 'suppliera', 'SupplierA' -> 'SupplierA'"""
    compact = name.strip().lower().replace(" ", "")
    return "Supplier" + compact[-1].upper()

status_fixes = {"Delievered": "Delivered", "Pendng": "Pending"}

changes = {col: 0 for col in columns}   # count of values changed per column
cleaned_rows = []

for row in rows:
    new_row = {}
    for col in columns:
        old = row[col]
        new = old.strip()                                   # strip spaces from every value

        if col == "supplier_name":
            new = standardise_supplier(new)
        elif col == "warehouse_destination":
            new = new.replace("WH_", "Warehouse_")
        elif col == "delivery_status":
            new = status_fixes.get(new, new)                # fix typos, leave others alone
        elif col in ("order_date", "delivery_date") and new != "":
            new = parse_date(new).isoformat()               # everything becomes YYYY-MM-DD

        if new != old:
            changes[col] += 1
        new_row[col] = new

    cleaned_rows.append(new_row)

print("VALUES CHANGED PER COLUMN")
for col in columns:
    if changes[col] > 0:
        print(f"  {col}: {changes[col]}")
print("Total values changed:", sum(changes.values()))

# Save a NEW file (never overwrite the original)
with open("supply_chain_standardised.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows(cleaned_rows)

# Reload and check the distinct values are now clean
with open("supply_chain_standardised.csv", newline="") as file:
    check = list(csv.DictReader(file))

print("\nRELOAD CHECK, rows:", len(check))
for col in ["supplier_name", "warehouse_destination", "delivery_status"]:
    print(f"  {col}: {sorted({r[col] for r in check})}")
print("  order_date values with '/':", sum(1 for r in check if "/" in r["order_date"]))
print("  delivery_date values with '/':", sum(1 for r in check if "/" in r["delivery_date"]))

VALUES CHANGED PER COLUMN
  order_date: 317
  supplier_name: 1945
  warehouse_destination: 276
  delivery_date: 296
  delivery_status: 43
Total values changed: 2877

RELOAD CHECK, rows: 3000
  supplier_name: ['SupplierA', 'SupplierB', 'SupplierC', 'SupplierD', 'SupplierE']
  warehouse_destination: ['Warehouse_1', 'Warehouse_2', 'Warehouse_3', 'Warehouse_4', 'Warehouse_5', 'Warehouse_6']
  delivery_status: ['Cancelled', 'Delayed', 'Delivered', 'In-Transit', 'Pending']
  order_date values with '/': 0
  delivery_date values with '/': 0


## Data Quality Report

| Issue | Rows affected | Fixed this week? | What should happen next |
|---|---|---|---|
| Supplier name spellings (15 variants of 5 suppliers) | 1,945 values | Yes | None, now SupplierA to SupplierE |
| Warehouse names (`WH_1` vs `Warehouse_1`) | 276 values | Yes | None, now Warehouse_1 to Warehouse_6 |
| Delivery status typos (`Delievered`, `Pendng`) | 43 values (22 + 21) | Yes | None |
| Mixed date formats in `order_date` | 317 values | Yes | None, all YYYY-MM-DD |
| Mixed date formats in `delivery_date` | 296 values | Yes | None, all YYYY-MM-DD |
| Missing `product_defects` | 154 | No | Decide in Week 7 whether to fill (e.g. with 0) or leave blank |
| Missing `return_status` | 146 | No | Check whether blank means "no return" or truly missing, then decide in Week 7 |
| Exact duplicate rows | 0 | n/a | None found |
| Category values with extra spaces | 0 | n/a | None found |
| Cost is zero or negative | 1 (ORD-1521) | No | Investigate the order, then correct or drop in Week 7 |
| Cost ≠ quantity × unit price | 154 | No | Find which value is wrong (cost, quantity or price) before fixing |
| Delivered before ordered, and actual days ≠ days between dates | 23 (same rows) | No | Check the source dates, then correct or drop in Week 7 |
| `on_time` disagrees with actual vs lead days | 54 | No | Recalculate `on_time` from the days columns in Week 7 |
| Orders with no delivery yet (Cancelled, In-Transit, Pending) | 107 | n/a | Expected, not an error. Skipped in the delivery checks |

In [6]:
import collections

# Challenge 1: validate_order returns a list of everything wrong with one order
def validate_order(row):
    problems = []
    if row["product_defects"] == "":
        problems.append("missing product_defects")
    if row["return_status"] == "":
        problems.append("missing return_status")

    cost = float(row["total_cost"])
    qty = float(row["quantity"])
    price = float(row["unit_price"])
    if cost <= 0:
        problems.append("cost zero or negative")
    if abs(qty * price - cost) > 1:
        problems.append("cost not qty x price")

    # No delivery yet, so skip the delivery checks
    if row["delivery_status"].strip() in ["Cancelled", "In-Transit", "Pending", "Pendng"]:
        return problems

    ordered = parse_date(row["order_date"])
    delivered = parse_date(row["delivery_date"])
    actual = int(float(row["actual_delivery_days"]))
    lead = float(row["lead_time_days"])
    if delivered < ordered:
        problems.append("delivered before ordered")
    if (delivered - ordered).days != actual:
        problems.append("actual days mismatch")
    late = actual > lead
    if (row["on_time"] == "Yes" and late) or (row["on_time"] == "No" and not late):
        problems.append("on_time disagrees")
    return problems

failed = [(row, validate_order(row)) for row in rows]
failed = [(row, problems) for row, problems in failed if problems]
print("Challenge 1: rows that fail:", len(failed))

# Challenge 2: which supplier has the most problem rows?
problems_by_supplier = collections.Counter(
    standardise_supplier(row["supplier_name"]) for row, p in failed)
orders_by_supplier = collections.Counter(
    standardise_supplier(row["supplier_name"]) for row in rows)
print("\nChallenge 2: problem rows by supplier")
for supplier, count in problems_by_supplier.most_common():
    print(f"  {supplier}: {count} of {orders_by_supplier[supplier]} orders "
          f"({count / orders_by_supplier[supplier] * 100:.1f}%)")

# Challenge 3: which month has the most problems?
problems_by_month = collections.Counter(parse_date(row["order_date"]).month for row, p in failed)
orders_by_month = collections.Counter(parse_date(row["order_date"]).month for row in rows)
print("\nChallenge 3: problem rows by order month")
for month in sorted(orders_by_month):
    count = problems_by_month[month]
    print(f"  Month {month:>2}: {count} of {orders_by_month[month]} orders "
          f"({count / orders_by_month[month] * 100:.1f}%)")

Challenge 1: rows that fail: 498

Challenge 2: problem rows by supplier
  SupplierA: 123 of 769 orders (16.0%)
  SupplierE: 114 of 647 orders (17.6%)
  SupplierB: 106 of 659 orders (16.1%)
  SupplierC: 80 of 492 orders (16.3%)
  SupplierD: 75 of 433 orders (17.3%)

Challenge 3: problem rows by order month
  Month  1: 41 of 253 orders (16.2%)
  Month  2: 38 of 233 orders (16.3%)
  Month  3: 48 of 260 orders (18.5%)
  Month  4: 49 of 269 orders (18.2%)
  Month  5: 49 of 237 orders (20.7%)
  Month  6: 41 of 235 orders (17.4%)
  Month  7: 38 of 259 orders (14.7%)
  Month  8: 36 of 256 orders (14.1%)
  Month  9: 29 of 246 orders (11.8%)
  Month 10: 48 of 270 orders (17.8%)
  Month 11: 40 of 238 orders (16.8%)
  Month 12: 41 of 244 orders (16.8%)


I found 11 types of data quality problems in the 3,000-row supply chain dataset. The most common was inconsistent supplier names, with 1,945 values written in the wrong form out of 15 spellings of just 5 suppliers. This week I standardised the text and dates: supplier names, warehouse names (WH_ to Warehouse_), two delivery status typos, and 613 dates in mixed formats, changing 2,877 values in total and saving them to supply_chain_standardised.csv. After reloading the file, there are 5 suppliers, 6 warehouses, 5 statuses and no mixed date formats. 498 rows (16.6%) still have at least one problem, which I deliberately did not change this week: 293 rows have blank product_defects or return_status, and 228 rows fail logical checks, including 154 where cost doesn’t equal quantity × price, 54 where on_time disagrees with the days, 23 with impossible delivery dates, and 1 negative cost. These need decisions in Week 7, and the problem rate is similar across suppliers (16 to 18%), so no single supplier is the cause.